In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys; sys.path.append("../../../"); sys.path.append("../../utils")
import pandas as pd

from ast import literal_eval

from pathlib import Path

from kb_graph import KBGraph
from kb_mapper import KBMapper, skill_pool_uris
from mapping_utils import pending, run_chunked

# KB to map the skills onto: sets the graph cache, the mapper's aliases and descriptions, the
# output files. The job title stays mapped to ESCO; `esco_occupation` is only the context.
KB = "ONET"
GRAPH_CACHE = {"ESCO": "occSkill_graph.pkl", "ONET": "onet_occSkill_graph.pkl"}[KB]

In [ ]:
preprocessed_job = (
    pd
    .read_json("mapping/preprocessed_jobs.json")
    .merge(
        pd.read_csv("mapping/occupation_map.csv")[["id_j", "esco_occupation"]],
        on="id_j",how="left"
    )
    .merge(
        pd.read_csv("mapping/translated.csv")[["id_j", "eng_description"]],
        on="id_j",how="left"
    )
    .merge(
        pd.read_csv("mapping/translated_skills.csv", converters={"eng_skills": literal_eval})[["id_j", "eng_skills"]],
        on="id_j",how="left"
    )
)[["id_j","esco_occupation", "eng_description", "eng_skills"]]
preprocessed_job.head(10)

In [ ]:
kb_g = KBGraph.load(Path("../../../cache") / GRAPH_CACHE)

all_skills_uri = skill_pool_uris(kb_g, min_level = 4)
all_skills = sorted(kb_g.get_label(i) for i in all_skills_uri)

assert all(kb_g.get_level(i) >= 4 for i in all_skills_uri)
assert all(kb_g.get_level(kb_g.get_uri(i, reduce="Down")) >= 4 for i in all_skills)

In [ ]:
MAPPER = KBMapper(all_skills, node="skill", processed_dir=Path("../../../sources") / KB / "processed", kb=KB)

In [ ]:
def extract_skills(row):
    """Map every skill of the offer onto the KB. Returns the aggregate row and N audit rows."""
    context = str({"job title:": row["esco_occupation"], "short description": ""})
    mapped = [(skill, MAPPER.map_skill(skill, context=context)) for skill in row["eng_skills"]]
    return (
        {"id_j": row["id_j"], "skills_list": row["eng_skills"],
         "esco_skills": sorted({r["kb_label"] for _, r in mapped if r["kb_label"]})},
        [{"id_j": row["id_j"], "raw_skill": skill, **r} for skill, r in mapped],
    )

In [ ]:
# The column is called `esco_skills` for every KB; 6_build_jobs.ipynb renames it.
AGG = (f"mapping/{KB}_skill_map.csv", ["id_j", "skills_list", "esco_skills"])
# The keys KBMapper._row emits, plus id_j/raw_skill.
DET = (f"mapping/{KB}_skill_map_detailed.csv",
       ["id_j", "raw_skill", "kb_label", "semantic_score", "semantic_rank",
        "candidate_list", "decision_source", "confidence", "mapping_reason"])

todo = pending(preprocessed_job, AGG, DET)

In [ ]:
run_chunked(todo, extract_skills, AGG, DET)